# Práctica 1 - Pipeline alternativo de Machine Learning para predicción de impago

## Objetivo
En este notebook se construye un pipeline alternativo completo de Machine Learning para predecir si un cliente devolverá su préstamo o no.

Pipeline alternativo:
- usamos `variables_withExperts.xlsx`, incorporando variables de expertos;
- imputamos numéricas con un valor constante `-1` y categóricas con `"missing"`;
- codificamos variables ordinales con `OrdinalEncoder` y el resto con `CountFrequencyEncoder`;
- escalamos con `RobustScaler`, más robusto frente a outliers;
- generamos variables nuevas basadas en ratios y agregaciones financieras;
- y filtramos con `VarianceThreshold` y `SelectKBest(mutual_info_classif)`.

Estas decisiones buscan construir un pipeline más compacto, interpretable y estable computacionalmente.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    precision_recall_curve,
    auc
)

from src.preprocessing.practica1_preprocessing import Practica1Preprocess
from src.filtering.practica1_filtering import Practica1Filtering

In [2]:
# Carga de datos

df_train = pd.read_csv("data/df_train_small.csv")
df_test = pd.read_csv("data/df_test_small.csv")

print("Dimensiones train:", df_train.shape)
print("Dimensiones test:", df_test.shape)

df_train.head()

Dimensiones train: (80000, 151)
Dimensiones test: (20000, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,86662190,NaN,8000.0,8000.0,8000.0,36 months,7.59,249.19,A,A3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,59210813,NaN,18000.0,18000.0,18000.0,36 months,11.53,593.83,B,B5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,23623421,NaN,6200.0,6200.0,6200.0,36 months,10.99,202.96,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,50203881,NaN,10200.0,10200.0,10200.0,36 months,7.89,319.12,A,A5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,90239836,NaN,15000.0,15000.0,15000.0,36 months,11.49,494.57,B,B5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Creación del target
y_train = (df_train["loan_status"] != "Fully Paid").astype(int)
y_test = (df_test["loan_status"] != "Fully Paid").astype(int)

print("Proporción de impago en train:", y_train.mean().round(4))
print("Proporción de impago en test:", y_test.mean().round(4))

Proporción de impago en train: 0.203
Proporción de impago en test: 0.1998


Tras eliminar la variable objetivo, los modelos seguían mostrando resultados anormalmente elevados.

Se detectó que el conjunto de datos incluía variables que contienen información posterior a la concesión del préstamo, como pagos totales, recoveries o variables de settlement y hardship.

Estas variables introducen data leakage, ya que no estarían disponibles en el momento de realizar la predicción.

Una vez eliminadas, los resultados del modelo se volvieron más realistas y coherentes con la dificultad del problema.

In [4]:
variables_df = pd.read_excel("data/variables_withExperts.xlsx")
print("Duplicadas en Excel:", variables_df.iloc[:, 0].duplicated().sum())
print("Columnas duplicadas en df_train:", df_train.columns.duplicated().sum())
print(df_train.columns[df_train.columns.duplicated()].tolist())

Duplicadas en Excel: 0
Columnas duplicadas en df_train: 0
[]


## Preprocesamiento

En esta práctica se ha implementado una clase `Practica1Preprocess` con varias diferencias:

- **Variables**: se usa `variables_withExperts.xlsx`, incluyendo variables de evaluación experta como `grade`, `sub_grade`, `int_rate`, `installment` y el rango FICO.
- **Imputación numérica**: se usa imputación constante con `-1`, como alternativa rápida y explícita frente a la imputación simple de la clase base.
- **Imputación categórica**: se usa imputación constante con `"missing"`, para representar ausencias como una categoría explícita.
- **Variables categóricas**:
  - `OrdinalEncoder` para `grade` y `sub_grade`, porque tienen orden natural;
  - `CountFrequencyEncoder` para el resto, evitando el crecimiento de dimensiones de OneHotEncoder.
- **Variables numéricas**: se usa `RobustScaler`, ya que las variables financieras suelen contener outliers.
- **Nuevas variables**: se crean variables de dominio como `fico_mean`, ratios relativos a ingresos, antigüedad del historial crediticio y binning de ingresos.

Todas estas transformaciones se ajustan únicamente con train y luego se aplican a train y test.

In [5]:
preprocessor = Practica1Preprocess(
    variables_path="data/variables_withExperts.xlsx"
)

X_train_prep = preprocessor.fit(df_train).transform(df_train)
X_test_prep = preprocessor.transform(df_test)

print("Dimensiones originales train:", df_train.shape)
print("Dimensiones tras preprocessing train:", X_train_prep.shape)

print("Dimensiones originales test:", df_test.shape)
print("Dimensiones tras preprocessing test:", X_test_prep.shape)

print("\nNaN en X_train_prep:", X_train_prep.isna().sum().sum())
print("NaN en X_test_prep:", X_test_prep.isna().sum().sum())

X_train_prep.head()

Dimensiones originales train: (80000, 151)
Dimensiones tras preprocessing train: (80000, 129)
Dimensiones originales test: (20000, 151)
Dimensiones tras preprocessing test: (20000, 129)

NaN en X_train_prep: 0
NaN en X_test_prep: 31087


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,...,initial_list_status,application_type,verification_status_joint,sec_app_earliest_cr_line,hardship_flag,hardship_type,hardship_reason,hardship_loan_status,disbursement_method,annual_inc_bin
0,0.522072,0.0,-0.327869,-0.327869,-0.327869,-0.825321,-0.376384,1.475914,0.701581,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.183152
1,0.061354,0.0,0.491803,0.491803,0.491803,-0.193910,0.660190,0.938681,-0.570138,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.183152
2,-0.535911,0.0,-0.475410,-0.475410,-0.475410,-0.280449,-0.515429,-0.749310,0.257959,1.0,...,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
3,-0.089810,0.0,-0.147541,-0.147541,-0.147541,-0.777244,-0.166055,-0.930961,1.324297,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
4,0.582115,0.0,0.245902,0.245902,0.245902,-0.200321,0.361646,0.227064,-0.010680,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.633696


## Filtrado de variables

En esta práctica se ha implementado una clase `Practica1Filtering` con una estrategia distinta:

1. limpieza de valores problemáticos (`NaN`, `inf`, `-inf`);
2. imputación residual con valor 0;
3. `VarianceThreshold`, para eliminar variables sin variación útil;
4. `SelectKBest(mutual_info_classif)`, para seleccionar las variables más relevantes respecto al target.

Se ha elegido esta combinación porque:
- reduce dimensionalidad,
- es fácil de interpretar,
- y la información mutua puede capturar relaciones no lineales entre una variable y la clase objetivo.

In [6]:
filtering = Practica1Filtering(
    variance_threshold=0.0,
    k_best=80
)

X_train_filt = filtering.fit(X_train_prep, y_train).transform(X_train_prep)
X_test_filt = filtering.transform(X_test_prep)

print("Dimensiones tras preprocessing train:", X_train_prep.shape)
print("Dimensiones tras filtering train:", X_train_filt.shape)

print("Dimensiones tras preprocessing test:", X_test_prep.shape)
print("Dimensiones tras filtering test:", X_test_filt.shape)

print("\nNaN en X_train_filt:", X_train_filt.isna().sum().sum())
print("NaN en X_test_filt:", X_test_filt.isna().sum().sum())

X_train_filt.head()

Dimensiones tras preprocessing train: (80000, 129)
Dimensiones tras filtering train: (80000, 80)
Dimensiones tras preprocessing test: (20000, 129)
Dimensiones tras filtering test: (20000, 80)

NaN en X_train_filt: 0
NaN en X_test_filt: 0


,id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,fico_range_low,fico_range_high,...,title,addr_state,earliest_cr_line,initial_list_status,application_type,verification_status_joint,hardship_type,hardship_reason,hardship_loan_status,annual_inc_bin
0,0.522072,-0.327869,-0.327869,-0.327869,-0.825321,-0.376384,1.475914,0.701581,0.750,0.750,...,-0.298011,-0.067220,-0.688,0.0,0.0,0.0,0.0,0.0,0.0,-0.183152
1,0.061354,0.491803,0.491803,0.491803,-0.193910,0.660190,0.938681,-0.570138,-0.750,-0.750,...,0.691293,-0.420124,-1.196,0.0,0.0,0.0,0.0,0.0,0.0,-0.183152
2,-0.535911,-0.475410,-0.475410,-0.475410,-0.280449,-0.515429,-0.749310,0.257959,-0.500,-0.500,...,0.000000,-0.094813,0.892,-1.0,0.0,0.0,0.0,0.0,0.0,0.000000
3,-0.089810,-0.147541,-0.147541,-0.147541,-0.777244,-0.166055,-0.930961,1.324297,1.250,1.250,...,0.691293,-0.336722,0.672,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
4,0.582115,0.245902,0.245902,0.245902,-0.200321,0.361646,0.227064,-0.010680,-0.375,-0.375,...,0.691293,-0.287137,-0.708,0.0,0.0,0.0,0.0,0.0,0.0,-0.633696


## Modelos

In [7]:
# Métricas
from sklearn.metrics import accuracy_score, precision_score, recall_score, precision_recall_curve, auc

def evaluate_model(model, X_train, y_train, X_test, y_test, name):
    # entrenar
    model.fit(X_train, y_train)

    # predecir clases
    y_pred = model.predict(X_test)

    # probabilidades para PR-AUC
    y_prob = model.predict_proba(X_test)[:, 1]

    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = auc(recall_vals, precision_vals)

    return {
        "Modelo": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision_default": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall_default": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "PR_AUC": pr_auc
    }

### Modelo 1 - Ensemble de árboles


Como modelo de tipo ensemble se ha seleccionado un `RandomForestClassifier`.

Este modelo:
- maneja bien datos tabulares,
- puede capturar relaciones no lineales,
- es robusto frente a ruido,
- y no requiere una normalización especialmente delicada para funcionar correctamente.

Además, se utiliza `class_weight="balanced"` para compensar el desbalanceo entre clases.

In [8]:
# Entrenamiento
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_results = evaluate_model(
    rf_model,
    X_train_filt,
    y_train,
    X_test_filt,
    y_test,
    "Random Forest"
)

rf_results

{'Modelo': 'Random Forest',
 'Accuracy': 0.8776,
 'Precision_default': 0.6354731502536295,
 'Recall_default': 0.9089316987740805,
 'PR_AUC': 0.8284397313720823}

### Modelo 2 - Máquina de Soporte Vectorial

Como segundo modelo se ha utilizado una SVM con kernel RBF.

Esta elección permite modelar relaciones no lineales entre variables. Se activa `probability=True` para poder calcular la métrica PR-AUC, tal y como pide el enunciado.

Dado que las SVM pueden ser costosas computacionalmente con muchos datos, se ha elegido una configuración razonable sin realizar una optimización exhaustiva de hiperparámetros.

In [9]:
# Muestra opcional para acelerar la SVM
sample_size = min(5000, len(X_train_filt))

X_train_svm = X_train_filt.sample(n=sample_size, random_state=42)
y_train_svm = y_train.loc[X_train_svm.index]

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42
)

svm_results = evaluate_model(
    svm_model,
    X_train_svm,
    y_train_svm,
    X_test_filt,
    y_test,
    "SVM (RBF)"
)

svm_results

{'Modelo': 'SVM (RBF)',
 'Accuracy': 0.79395,
 'Precision_default': 0.19607843137254902,
 'Recall_default': 0.010007505629221916,
 'PR_AUC': 0.12783979126019687}

### Modelo 3 - Red neuronal


Como tercer modelo se ha elegido un `MLPClassifier`, que representa la familia de redes neuronales pedida en la práctica.

Este modelo puede capturar relaciones complejas entre variables, aunque su rendimiento puede depender de forma sensible de la escala de los datos y de la configuración de hiperparámetros. Se ha elegido una arquitectura moderada para evitar tiempos de entrenamiento excesivos y reducir el riesgo de sobreajuste.

In [10]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.001,
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    random_state=42
)

mlp_results = evaluate_model(
    mlp_model,
    X_train_filt,
    y_train,
    X_test_filt,
    y_test,
    "MLP"
)

mlp_results

{'Modelo': 'MLP',
 'Accuracy': 0.90425,
 'Precision_default': 0.7699688796680498,
 'Recall_default': 0.7428071053289967,
 'PR_AUC': 0.8304771464700078}

## Tabla resultados

In [11]:
base_results = {
    "Modelo": "Modelo base (FICO)",
    "Accuracy": 0.72,
    "Precision_default": 0.26,
    "Recall_default": 0.24,
    "PR_AUC": np.nan
}

results_df = pd.DataFrame([
    base_results,
    rf_results,
    svm_results,
    mlp_results
])

results_df

,Modelo,Accuracy,Precision_default,Recall_default,PR_AUC
0,Modelo base (FICO),0.72000,0.260000,0.240000,NaN
1,Random Forest,0.87760,0.635473,0.908932,0.828440
2,SVM (RBF),0.79395,0.196078,0.010008,0.127840
3,MLP,0.90425,0.769969,0.742807,0.830477


In [12]:
results_df_rounded = results_df.copy()

for col in ["Accuracy", "Precision_default", "Recall_default", "PR_AUC"]:
    results_df_rounded[col] = results_df_rounded[col].round(4)

results_df_rounded

,Modelo,Accuracy,Precision_default,Recall_default,PR_AUC
0,Modelo base (FICO),0.7200,0.2600,0.2400,NaN
1,Random Forest,0.8776,0.6355,0.9089,0.8284
2,SVM (RBF),0.7940,0.1961,0.0100,0.1278
3,MLP,0.9042,0.7700,0.7428,0.8305


Dado que el problema está desbalanceado, la métrica más relevante no es únicamente la accuracy, sino especialmente el recall y el PR-AUC, ya que reflejan mejor la capacidad del modelo para detectar la clase minoritaria (impago).

## Comparación con modelo base

In [13]:
base_acc = base_results["Accuracy"]
base_prec = base_results["Precision_default"]
base_rec = base_results["Recall_default"]

comparison_rows = []

for _, row in results_df.iloc[1:].iterrows():
    comparison_rows.append({
        "Modelo": row["Modelo"],
        "Delta_Accuracy_vs_base": row["Accuracy"] - base_acc,
        "Delta_Precision_vs_base": row["Precision_default"] - base_prec,
        "Delta_Recall_vs_base": row["Recall_default"] - base_rec
    })

comparison_df = pd.DataFrame(comparison_rows).round(4)
comparison_df

,Modelo,Delta_Accuracy_vs_base,Delta_Precision_vs_base,Delta_Recall_vs_base
0,Random Forest,0.1576,0.3755,0.6689
1,SVM (RBF),0.0740,-0.0639,-0.2300
2,MLP,0.1843,0.5100,0.5028


El modelo base, construido únicamente a partir del FICO score, obtiene:

- Accuracy: 72%
- Precision: 26%
- Recall: 24%

### Random Forest
- Mejora significativamente en todas las métricas:
  - Accuracy: +15.8 puntos
  - Precision: +0.38
  - Recall: +0.67
- Destaca especialmente en recall (0.91), lo que indica que detecta la mayoría de los impagos.
- Es un modelo adecuado si el objetivo principal es minimizar el riesgo de no detectar clientes que van a impagar.

### SVM (RBF)
- Mejora ligeramente en accuracy, pero empeora claramente en precision y recall.
- Presenta un recall muy bajo (0.01), lo que indica que apenas detecta la clase de impago.
- Esto puede deberse al desbalanceo de clases y a la sensibilidad del SVM a la elección de hiperparámetros.
- En este caso, no es un modelo adecuado para el problema.

### MLP (Red neuronal)
- Es el modelo con mejor rendimiento global:
  - Accuracy: +18.4 puntos
  - Precision: +0.51
  - Recall: +0.50
- Presenta el mejor equilibrio entre precision y recall.
- Además, obtiene el mayor PR-AUC (0.83), lo que indica un buen rendimiento en la detección de la clase minoritaria.

## Conclusión

El modelo MLP es el mejor en términos globales, ya que ofrece el mejor equilibrio entre todas las métricas.

El Random Forest también es una opción muy sólida, especialmente si se prioriza el recall (detección de impagos).

El modelo SVM, en cambio, no resulta adecuado en este contexto debido a su bajo rendimiento en la clase minoritaria.

## Interpretación final

A nivel bancario, no basta con obtener una buena accuracy global, ya que el objetivo principal del problema es detectar correctamente los casos de impago.

Por ello, las métricas más relevantes son:
- **recall de la clase de impago**, para identificar el mayor número posible de clientes con riesgo;
- **precision de la clase de impago**, para evitar marcar como morosos a demasiados clientes solventes;
- **PR-AUC**, porque resume el rendimiento sobre la clase minoritaria.

Analizando los resultados obtenidos:

- El **Random Forest** destaca especialmente en recall (~0.91), lo que indica que es capaz de detectar la gran mayoría de los impagos. Sin embargo, su precision es menor que la del MLP, por lo que genera más falsos positivos.

- El modelo **MLP (red neuronal)** ofrece el mejor equilibrio global, con alta accuracy (~0.90), buena precision (~0.77) y un recall también elevado (~0.74). Además, obtiene el mayor PR-AUC, lo que indica un buen rendimiento general en la clase minoritaria.

- La **SVM** no funciona bien en este problema: aunque tiene una accuracy aceptable, su recall es prácticamente nulo, lo que significa que no detecta los impagos. Esto la hace poco útil en un contexto real.

En este contexto, el mejor modelo no es simplemente el que maximiza la accuracy, sino aquel que ofrece un equilibrio adecuado entre precision y recall para la clase de impago. En este caso, el MLP es la opción más equilibrada, mientras que el Random Forest sería preferible si se prioriza maximizar la detección de riesgo.

## Conclusión

En esta práctica se ha construido un pipeline alternativo completo para la predicción de impago, manteniendo la estructura general trabajada en clase pero introduciendo decisiones propias en todas las fases: preprocesamiento, generación de variables, filtrado y modelado.

Uno de los aspectos más importantes del trabajo ha sido la detección y corrección de **data leakage**. Inicialmente, se obtuvieron resultados prácticamente perfectos (accuracy cercana a 1), lo cual indicaba un problema en el pipeline. Tras analizar las variables, se identificó que se estaban incluyendo variables que contenían información posterior a la concesión del préstamo (como pagos realizados, recoveries o variables de settlement), así como la propia variable objetivo.

Estas variables no estarían disponibles en un escenario real de predicción, por lo que su uso introduce leakage y hace que el modelo “vea el futuro”. Una vez eliminadas correctamente, los resultados se volvieron más realistas y coherentes con la dificultad del problema.

A nivel de modelado, se han comparado tres enfoques distintos:
- un ensemble de árboles (`RandomForestClassifier`),
- una SVM con kernel RBF,
- y una red neuronal (`MLPClassifier`).

Los resultados muestran que:
- el **MLP** es el modelo con mejor rendimiento global,
- el **Random Forest** destaca en la detección de impagos (alto recall),
- y la **SVM** no resulta adecuada en este contexto.

En conjunto, esta práctica demuestra la importancia de:
- un buen diseño del preprocesamiento,
- la correcta selección de variables,
- y, especialmente, la identificación de posibles fugas de información.

Más allá de las métricas, el trabajo refleja cómo decisiones aparentemente pequeñas en el pipeline pueden tener un impacto muy significativo en el rendimiento y la validez del modelo.